In [0]:
#print ("Hello")

In [0]:
# List all secret scopes
#dbutils.secrets.listScopes()



In [0]:
# List all keys in the 'optumscope' scope
#dbutils.secrets.list("optumscope")

In [0]:
%run /Workspace/Users/az_data_eng_25@outlook.com/genric_connectors

In [0]:
%run /Workspace/Users/az_data_eng_25@outlook.com/generic_transformations

In [0]:
spark_connector()

'Connected to adls'

In [0]:
%python
spark.conf.set(
    "fs.azure.account.key.optumadlssg.dfs.core.windows.net",
    dbutils.secrets.get(scope = 'optumscope', key = 'adlskey'))

In [0]:
#display(dbutils.fs.ls("abfss://optum@optumadlssg.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/,bronze/,0,1757754080000
abfss://optum@optumadlssg.dfs.core.windows.net/gold/,gold/,0,1757754097000
abfss://optum@optumadlssg.dfs.core.windows.net/silver/,silver/,0,1757754088000


In [0]:
#display(dbutils.fs.ls("abfss://optum@optumadlssg.dfs.core.windows.net/bronze"))

path,name,size,modificationTime
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/Claims.json,Claims.json,16385,1757826153000
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/Patient_records.csv,Patient_records.csv,5110,1757826143000
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/disease.csv,disease.csv,1489,1757826143000
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/group.csv,group.csv,4390,1757826143000
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/hospital.csv,hospital.csv,1328,1757826143000
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/subgroup.csv,subgroup.csv,561,1757826143000
abfss://optum@optumadlssg.dfs.core.windows.net/bronze/subscriber.csv,subscriber.csv,12061,1757826143000


In [0]:
#hos_df = spark.read.csv("abfss://optum@optumadlssg.dfs.core.windows.net/bronze/hospital.csv",inferSchema=True,header=True)

In [0]:
pat_df= read_bronze_csv_data("Patient_records")

In [0]:
display(pat_df)

Patient_id,Patient_name,patient_gender,patient_birth_date,patient_phone,disease_name,city,hospital_id
187158,Harbir,Female,1924-06-30,+91 0112009318,Galactosemia,Rourkela,H1001
112766,Brahmdev,Female,1948-12-20,+91 1727749552,Bladder cancer,Tiruvottiyur,H1016
199252,Ujjawal,Male,1980-04-16,+91 8547451606,Kidney cancer,Berhampur,H1009
133424,Ballari,Female,1969-09-25,+91 0106026841,Suicide,Bihar Sharif,H1017
172579,Devnath,Female,1946-05-01,+91 1868774631,Food allergy,Bidhannagar,H1019
171320,Atasi,Male,1967-10-02,+91 9747336855,Whiplash,Amravati,H1013
107794,Manish,Male,1967-06-06,+91 4354294043,Sunbathing,Panvel,H1004
130339,Aakar,Female,1925-03-05,+91 2777633911,Drug consumption,Bihar Sharif,H1000
110377,Gurudas,Male,1945-05-06,+91 1232859381,Dengue,Kamarhati,H1001
149367,null,Male,1925-06-12,+91 1780763280,Head banging,Bangalore,H1013


In [0]:
df_row_columns_cnt(pat_df)

(70, 8)

In [0]:
#hos_df=hos_df.replace("NaN",None)

In [0]:
df_missing_value_cnt(pat_df)

column_name,missing_count
Patient_id,0
Patient_name,17
patient_gender,0
patient_birth_date,0
patient_phone,2
disease_name,0
city,0
hospital_id,0


In [0]:
missing_value_perc(pat_df)

Patient_id,Patient_name,patient_gender,patient_birth_date,patient_phone,disease_name,city,hospital_id
0.0,0.24285714285714285,0.0,0.0,0.02857142857142857,0.0,0.0,0.0


In [0]:
check_duplicate(pat_df)

No duplicate


In [0]:
display(pat_df)

Patient_id,Patient_name,patient_gender,patient_birth_date,patient_phone,disease_name,city,hospital_id
187158,Harbir,Female,1924-06-30,+91 0112009318,Galactosemia,Rourkela,H1001
112766,Brahmdev,Female,1948-12-20,+91 1727749552,Bladder cancer,Tiruvottiyur,H1016
199252,Ujjawal,Male,1980-04-16,+91 8547451606,Kidney cancer,Berhampur,H1009
133424,Ballari,Female,1969-09-25,+91 0106026841,Suicide,Bihar Sharif,H1017
172579,Devnath,Female,1946-05-01,+91 1868774631,Food allergy,Bidhannagar,H1019
171320,Atasi,Male,1967-10-02,+91 9747336855,Whiplash,Amravati,H1013
107794,Manish,Male,1967-06-06,+91 4354294043,Sunbathing,Panvel,H1004
130339,Aakar,Female,1925-03-05,+91 2777633911,Drug consumption,Bihar Sharif,H1000
110377,Gurudas,Male,1945-05-06,+91 1232859381,Dengue,Kamarhati,H1001
149367,null,Male,1925-06-12,+91 1780763280,Head banging,Bangalore,H1013


In [0]:
#patient specific trnansformations
pat_df= pat_df.replace({'Patient_name':"Vistor/NA"})
pat_df= pat_df.drop("Patient_phone")
pat_df = pat_df.withColumn("patient_age",round(date_diff(current_date(),to_date(col("patient_birth_date"),"MM-dd-yyyy"))/365))
pat_df= pat_df.drop("patient_birth_date")


In [0]:
display(pat_df)

Patient_id,Patient_name,patient_gender,disease_name,city,hospital_id,patient_age
187158,Harbir,Female,Galactosemia,Rourkela,H1001,101.0
112766,Brahmdev,Female,Bladder cancer,Tiruvottiyur,H1016,77.0
199252,Ujjawal,Male,Kidney cancer,Berhampur,H1009,45.0
133424,Ballari,Female,Suicide,Bihar Sharif,H1017,56.0
172579,Devnath,Female,Food allergy,Bidhannagar,H1019,79.0
171320,Atasi,Male,Whiplash,Amravati,H1013,58.0
107794,Manish,Male,Sunbathing,Panvel,H1004,58.0
130339,Aakar,Female,Drug consumption,Bihar Sharif,H1000,101.0
110377,Gurudas,Male,Dengue,Kamarhati,H1001,80.0
149367,null,Male,Head banging,Bangalore,H1013,100.0


In [0]:
write2silver(pat_df,"Patient_s.csv")

Patient_id,Patient_name,patient_gender,disease_name,city,hospital_id,patient_age
187158,Harbir,Female,Galactosemia,Rourkela,H1001,101.0
112766,Brahmdev,Female,Bladder cancer,Tiruvottiyur,H1016,77.0
199252,Ujjawal,Male,Kidney cancer,Berhampur,H1009,45.0
133424,Ballari,Female,Suicide,Bihar Sharif,H1017,56.0
172579,Devnath,Female,Food allergy,Bidhannagar,H1019,79.0
171320,Atasi,Male,Whiplash,Amravati,H1013,58.0
107794,Manish,Male,Sunbathing,Panvel,H1004,58.0
130339,Aakar,Female,Drug consumption,Bihar Sharif,H1000,101.0
110377,Gurudas,Male,Dengue,Kamarhati,H1001,80.0
149367,null,Male,Head banging,Bangalore,H1013,100.0


CSV File written sucessfully to silver path


In [0]:
check_duplicate(pat_df)

No duplicate


In [0]:
display(pat_df.select("*").filter(col("Patient_id").isin([134184,121783])))

Patient_id,Patient_name,patient_gender,disease_name,city,hospital_id,patient_age
134184,Prakash,Female,Flu,Kottayam,H1001,102.0
121783,Paridhi,Female,Bladder cancer,Jabalpur,H1013,67.0
